# 10. Bias in the data: heart-failure death

Same `ebdai` workflow as the Titanic demo, on the heart-failure table from [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025). The sensitive attribute is `sex` (0 = female, 1 = male); the label is `death`.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS, eval_tools
from ebdai import (
    features_and_target, load_heart_failure, outcome_rates_by_group,
    plot_outcome_rates, plot_winning_rules_by_group, fairness_report,
    parse_printed_rules, winning_rules_by_group,
)

frame, sensitive = load_heart_failure()
X, y = features_and_target(frame, 'death')
print(X.dtypes)
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Death rate by sex')

Fit a short-budget fuzzy classifier and inspect group-wise rule firings.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=6, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))
counts = winning_rules_by_group(
    clf, X_test, X_test[sensitive],
    rule_texts=parse_printed_rules(report or ''),
)
plot_winning_rules_by_group(counts, title='Winning heart-failure rules by sex')